# EDA : IRA & CHIPS Act et investissement industriel américain

Question : les deux lois d'août 2022 ont-elles fait décoller l'investissement industriel aux États-Unis ?
Sauf mention contraire, les graphes montrent un point par année. Le trait pointillé marque 2022, l'année du traitement, et le dernier point, creux, correspond à 2026, année encore incomplète.

In [1]:
import sys
from datetime import date

import polars as pl

sys.path.append('../src')
import data_processing as dp
from plot_functions import set_theme, plot_years, plot_treated_vs_controls, plot_small_multiples, plot_spaghetti,plot_monthly, plot_ranking

set_theme()

## 1. Données

In [2]:
data = dp.load_processed()
dp.overview(data)

fichier,lignes,colonnes,début,fin,% nul
str,i64,i64,str,str,f64
"""census_construction_real_month…",200,18,"""2010-01-01""","""2026-08-01""",1.3
"""cim_state_investment_share_gdp""",51,4,"""-""","""-""",0.0
"""cim_totals_quarterly""",34,3,"""2018-01-01""","""2026-04-01""",0.0
"""cim_us_clean_investment_quarte…",34,42,"""2018-01-01""","""2026-04-01""",24.1
"""did_panel_monthly""",1800,6,"""2010-01-01""","""2026-08-01""",0.0
"""fred_monthly""",201,19,"""2010-01-01""","""2026-09-01""",0.2
"""oecd_donor_fit""",20,4,"""-""","""-""",0.0
"""oecd_gfcf_index2019_quarterly""",66,22,"""2010-01-01""","""2026-04-01""",0.0


Les fichiers lus ici sont produits à partir de `data/raw/` par [`src/data_processing.py`](../src/data_processing.py), qui les écrit dans `data/processed/` (`uv run python src/data_processing.py`).
Les montants Census et FRED sont déflatés par l'indice des prix des matériaux de construction (`WPUSI012011`), qui a flambé en 2021-2022, et exprimés en dollars de 2019.
Le secteur traité est la construction d'usines électronique et électrique (`mfg_computer_electronic_electrical`). Les autres sous-secteurs manufacturiers servent de secteurs de contrôle, et le panel DiD y ajoute les bureaux et les commerces.
Pour l'OCDE, l'investissement de chaque pays est ramené en base 100 = 2019, et les 20 pays autres que les États-Unis forment le pool de donneurs du synthetic control.

In [3]:
fred = data['fred_monthly']
census = data['census_construction_real_monthly']
oecd = data['oecd_gfcf_index2019_quarterly']
cim = data['cim_totals_quarterly']
MFG = dp.mfg_sectors(census)
TREATED = dp.TREATED_SECTOR
CONTROLS = [c for c in MFG if c != TREATED]

## 2. Construction d'usines

In [4]:
plot_years(fred, 'date', ['TLMFGCONS', 'TLMFGCONS_real'], scale=1000,
           title="Dépenses de construction d'usines aux États-Unis",
           subtitle='Moyenne annuelle, Md$ en rythme annuel', ytitle='Md$')

En dollars constants, les dépenses passent de **64 Md\$ en 2021 à 175 Md\$ en 2024** (×2,7), puis reculent en 2025 et 2026.

In [5]:
plot_small_multiples(census, 'date', MFG, highlight=TREATED,
                     title='Construction par sous-secteur manufacturier',
                     subtitle='Moyenne annuelle, M$ de 2019 ; chaque graphe a sa propre échelle')

Le boom vient presque entièrement de l'électronique et de l'équipement électrique (semi-conducteurs, batteries). Les autres sous-secteurs restent stables.

## 3. Secteur traité et secteurs de contrôle

In [6]:
plot_treated_vs_controls(census, 'date', TREATED, CONTROLS,
                         title='Secteur traité et secteurs de contrôle',
                         subtitle='Construction réelle, base 100 = 2019')

Les deux groupes évoluent de façon comparable en 2019-2021. En 2024, le secteur traité est multiplié par 10 alors que les secteurs de contrôle gagnent environ 45 %.
Avant 2019, la série traitée fluctue fortement, ce qui fragilise l'hypothèse de tendances parallèles sur une fenêtre longue.

In [7]:
plot_monthly(census.with_columns(pl.mean_horizontal(CONTROLS).alias('controls')), 'date', [TREATED, 'controls'],
             date(2020, 1, 1), date(2023, 12, 1), highlight=TREATED,
             title='Secteur traité et secteurs de contrôle, 2020-2023',
             subtitle='Mensuel, M$ de 2019 en rythme annuel', ytitle='M$ de 2019')

En données mensuelles, la hausse démarre **fin 2021**, avant les lois. La construction du secteur traité passe de 8 020 M\$ en juin 2021 à 22 649 M\$ en juin 2022 (×2,8), avec un saut en octobre 2021 (de 8 369 à 12 967 M\$), tandis que la moyenne des secteurs de contrôle reste stable (de 6 653 à 7 016 M\$).
Deux explications sont possibles. La première est l'anticipation, puisque le CHIPS Act est débattu au Congrès dès 2021 et qu'Intel et TSMC annoncent leurs usines avant le vote. La seconde est un choc propre au secteur, la pénurie de puces de 2021.
Dans les deux cas, l'hypothèse de tendances parallèles avec un traitement daté d'août 2022 est menacée.

## 4. Comparaison entre pays

In [8]:
plot_spaghetti(oecd, 'quarter', 'USA',
               title="Investissement en bâtiments non résidentiels, États-Unis et 20 pays de l'OCDE",
               subtitle='Volume, base 100 = 2019')

À l'échelle du pays, les États-Unis restent dans le faisceau des autres pays. L'effet est dilué dans un agrégat qui inclut bureaux et infrastructures.

In [9]:
data['oecd_donor_fit']

pays,corr_pre,rmse_pre,indice_post
str,f64,f64,f64
"""IRL""",0.67,10.06,78.07
"""ESP""",0.55,4.96,112.67
"""DEU""",0.49,5.6,93.01
"""FIN""",0.46,6.56,91.1
"""SWE""",0.45,8.81,102.68
…,…,…,…
"""AUS""",-0.04,15.73,112.83
"""ISR""",-0.04,15.23,116.59
"""NOR""",-0.08,9.82,117.77


De 2015-T1 à 2022-T2, l'Espagne (corrélation 0,55, RMSE 4,96), l'Allemagne (0,49 et 5,6), la Finlande (0,46) et la Suède (0,45) suivent le mieux les États-Unis et sont des donneurs naturels pour le synthetic control. L'Irlande a la meilleure corrélation (0,67) mais un RMSE élevé (10,06), signe d'une série très volatile.
Les pays à corrélation négative sont à écarter, en particulier le Danemark (−0,52) et l'Italie (−0,31, indice moyen de 183 depuis 2024), dont la trajectoire diverge nettement.

## 5. Usines clean tech

In [10]:
plot_years(cim, 'quarter', ['annoncé', 'réalisé'], how='sum',
           title='Usines clean tech, investissements annoncés et réalisés',
           subtitle='Total annuel, Md$ 2024 (années complètes)', ytitle='Md$')

Les annonces culminent en 2022 à **91 Md\$**, l'année de l'IRA. Les dépenses réalisées suivent avec un à deux ans de retard et culminent à 49 Md\$ en 2024.

In [11]:
plot_ranking(data['cim_state_investment_share_gdp'], 'state', 'pct_gdp_manufacturing', n=15,
             title='Usines clean tech, 15 États les plus exposés',
             subtitle="Investissement manufacturier clean tech des 4 derniers trimestres, % du PIB de l'État",
             xtitle='% du PIB')

L'investissement se concentre dans quelques États. Le Nevada (0,99 % du PIB) et le Kentucky (0,77 %) devancent la Caroline du Sud, la Géorgie et le Tennessee, puis le Midwest (Indiana, Kansas, Michigan). Cette part pourrait servir de mesure d'exposition au traitement par État, dans un DiD à traitement continu.

## 6. Facteurs de confusion

In [12]:
plot_small_multiples(fred, 'date', ['TLMFGCONS_real', 'FEDFUNDS', 'GS10', 'WPUSI012011', 'IPG3344S', 'CES3133440001'],
                     title='Construction, taux, prix et effets en aval',
                     subtitle='Moyenne annuelle ; chaque graphe a sa propre échelle')

Le boom a lieu pendant la hausse des taux, ce qui tend plutôt à sous-estimer l'effet.
La production de semi-conducteurs augmente, mais l'emploi du secteur bouge peu.

## 7. À retenir

- **DiD** : secteur traité contre les autres sous-secteurs manufacturiers, en log.
- **Event study** : estimer l'effet mois par mois autour d'août 2022 pour voir la dynamique et d'éventuels effets avant les lois.
- **Anticipation** : tester une date de traitement plus précoce (mi-2021).
- **Synthetic control** : effet probablement faible à l'échelle du pays, l'agrégat étant trop large.
- **Donneurs** : Espagne, Allemagne, Finlande et Suède (meilleur ajustement avant 2022, tableau `oecd_donor_fit`), en écartant le Danemark et l'Italie.
- **Confusion** : contrôler les taux d'intérêt et le prix des matériaux.